# class ColumnMapper(Wrapping)

## `__init__`
```python
def __init__(self, wrapper: ArrayWrapper, col_arr: tp.Array1d, **kwargs) -> None:
    Wrapping.__init__(
        self,
        wrapper,
        col_arr=col_arr,
        **kwargs
    )
    self._wrapper = wrapper
    self._col_arr = col_arr
```

## `col_range`
使用此函数时，`self.col_arr` 必须是排好序的，返回 `self.col_arr` 的范围索引。
- 例如：`[0, 0, 0, 1, 1, 2] ——> [[0 3] [3 5] [5 6]]`
```python
@cached_property
def col_range(self) -> tp.ColRange:

    return nb.col_range_nb(self.col_arr, len(self.wrapper.columns))
```

## `col_map`
返回 `self.col_arr` 的列映射索引：
- 例如：`[2, 0, 1, 0, 2, 1, 0] ——> Tuple([1, 3, 6, 2, 5, 0, 4], [3, 2, 2])`
```python
@cached_property
def col_map(self) -> tp.ColMap:
    return nb.col_map_nb(self.col_arr, len(self.wrapper.columns))
```

## `_col_idxs_meta`
使用参数 `col_idxs` 选择 `self.col_arr`，返回对应索引和对应的新列数组。
- 例如 `self.col_arr = [0, 0, 0, 1, 1, 2]` 和 `col_idxs = [0, 2]`，返回 `Tuple([0, 1, 2, 5],  [0, 0, 0, 1] )`
```python
def _col_idxs_meta(self, col_idxs: tp.Array1d) -> tp.Tuple[tp.Array1d, tp.Array1d]:
    if self.is_sorted():
        new_indices, new_col_arr = nb.col_range_select_nb(self.col_range, to_1d_array(col_idxs))
    else:
        new_indices, new_col_arr = nb.col_map_select_nb(self.col_map, to_1d_array(col_idxs))
    return new_indices, new_col_arr
```

## is_sorted
```python
@cached_method
def is_sorted(self) -> bool:
    return nb.is_col_sorted_nb(self.col_arr)
```

## 例子

In [21]:
import numpy as np
import pandas as pd
import vectorbt as vbt
from vectorbt.records.col_mapper import ColumnMapper

# 1. 创建基础数据
dates = pd.date_range('2023-01-01', periods=10, freq='D')
stocks = ['AAPL', 'GOOGL', 'MSFT']
wrapper = vbt.ArrayWrapper(index=dates, columns=stocks, ndim=2, freq='1D')

print("创建的ArrayWrapper信息：")
print(f"索引（时间）: {wrapper.index[:3]}...{wrapper.index[-1]}")
print(f"列名（股票）: {wrapper.columns.tolist()}")
print(f"数组形状: {wrapper.shape}")
print()

# 2. 创建两种类型的列数组
sorted_col_arr = np.array([0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2])
unsorted_col_arr = np.array([2, 0, 1, 0, 2, 1, 0, 2, 1, 0, 2, 1])

print("数据数组:")
print(f"排序的列数组: {sorted_col_arr}")
print(f"未排序的列数组: {unsorted_col_arr}")
print()

# 3. 创建ColumnMapper
sorted_mapper = ColumnMapper(wrapper, sorted_col_arr)
unsorted_mapper = ColumnMapper(wrapper, unsorted_col_arr)
print()

创建的ArrayWrapper信息：
索引（时间）: DatetimeIndex(['2023-01-01', '2023-01-02', '2023-01-03'], dtype='datetime64[ns]', freq='D')...2023-01-10 00:00:00
列名（股票）: ['AAPL', 'GOOGL', 'MSFT']
数组形状: (10, 3)

数据数组:
排序的列数组: [0 0 0 0 1 1 1 2 2 2 2 2]
未排序的列数组: [2 0 1 0 2 1 0 2 1 0 2 1]




### get_col_arr
```python
@cached_method
def get_col_arr(self, group_by: tp.GroupByLike = None) -> tp.Array1d:
    group_arr = self.wrapper.grouper.get_groups(group_by=group_by)
    if group_arr is not None:
        col_arr = group_arr[self.col_arr]
    else:
        col_arr = self.col_arr
    return col_arr
```

In [ ]:
# =================== get_col_arr 方法演示 ===================
print("=== get_col_arr 方法演示 ===")

group_by = ['科技股', '科技股', '其他股票']
print(f"分组方案: {dict(zip(stocks, group_by))}")

print("\n原始列数组 vs 分组后列数组:")
print(f"排序数据   - 原始: {sorted_mapper.get_col_arr(group_by=None)}")
print(f"排序数据   - 分组: {sorted_mapper.get_col_arr(group_by=group_by)}")
print(f"未排序数据 - 原始: {unsorted_mapper.get_col_arr(group_by=None)}")
print(f"未排序数据 - 分组: {unsorted_mapper.get_col_arr(group_by=group_by)}")
print()

### get_col_range
```python
@cached_method
def get_col_range(self, group_by: tp.GroupByLike = None) -> tp.ColRange:
    if not self.wrapper.grouper.is_grouped(group_by=group_by):
        return self.col_range
    col_arr = self.get_col_arr(group_by=group_by)
    columns = self.wrapper.get_columns(group_by=group_by)
    return nb.col_range_nb(col_arr, len(columns))
```

In [ ]:
print("=== get_col_range 方法演示（修正版）===")

# 获取col_range并正确解析其结构
sorted_col_range = sorted_mapper.get_col_range(group_by=None)
sorted_grouped_range = sorted_mapper.get_col_range(group_by=group_by)

print(f"原始列范围类型: {type(sorted_col_range)}")
print(f"原始列范围: {sorted_col_range}")
print(f"分组列范围: {sorted_grouped_range}")

# 根据实际的数据结构来解析col_range
# col_range通常是一个二维数组，其中每行包含[start, end]或者是其他结构
print("\n正确解析列范围索引:")

if hasattr(sorted_col_range, 'shape') and len(sorted_col_range.shape) == 2:
    # 如果是二维数组，每行可能是[start, end]
    print("col_range是二维数组结构:")
    for i, range_info in enumerate(sorted_col_range):
        if i < len(stocks):
            stock_name = stocks[i]
            if len(range_info) >= 2:
                start, end = range_info[0], range_info[1]
                length = end - start
                print(f"  {stock_name}(列{i}): 位置{start}到{end-1} (共{length}条记录)")
            else:
                print(f"  {stock_name}(列{i}): {range_info}")
elif hasattr(sorted_col_range, '__iter__'):
    # 如果是其他可迭代结构
    print("col_range的详细结构:")
    for i, item in enumerate(sorted_col_range):
        if i < len(stocks):
            print(f"  {stocks[i]}(列{i}): {item}")
        if i >= 10:  # 限制输出数量
            print("  ...")
            break

# 对分组后的范围做同样的解析
print(f"\n分组后的列范围结构:")
if hasattr(sorted_grouped_range, 'shape') and len(sorted_grouped_range.shape) == 2:
    group_names = ['科技股', '其他股票']
    for i, range_info in enumerate(sorted_grouped_range):
        if i < len(group_names):
            group_name = group_names[i]
            if len(range_info) >= 2:
                start, end = range_info[0], range_info[1]
                length = end - start
                print(f"  {group_name}(组{i}): 位置{start}到{end-1} (共{length}条记录)")
print()

### get_col_map
```python
@cached_method
def get_col_map(self, group_by: tp.GroupByLike = None) -> tp.ColMap:
    if not self.wrapper.grouper.is_grouped(group_by=group_by):
        return self.col_map
    col_arr = self.get_col_arr(group_by=group_by)
    columns = self.wrapper.get_columns(group_by=group_by)
    return nb.col_map_nb(col_arr, len(columns))
```

In [22]:
# =================== get_col_map 方法演示 ===================
print("=== get_col_map 方法演示 ===")

# col_map的结构通常是(col_idxs, col_lens)
sorted_col_map = sorted_mapper.get_col_map(group_by=None)
print(sorted_col_map)
sorted_col_map = sorted_mapper.get_col_map(group_by=group_by)
print(sorted_col_map)
unsorted_grouped_map = unsorted_mapper.get_col_map(group_by=None)
print(unsorted_grouped_map)
unsorted_grouped_map = unsorted_mapper.get_col_map(group_by=group_by)
print(unsorted_grouped_map)

=== get_col_map 方法演示 ===
(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]), array([4, 3, 5]))
(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11]), array([7, 5]))
(array([ 1,  3,  6,  9,  2,  5,  8, 11,  0,  4,  7, 10]), array([4, 4, 4]))
(array([ 1,  2,  3,  5,  6,  8,  9, 11,  0,  4,  7, 10]), array([8, 4]))
